# KBase MCP API Explorer

This notebook lets you interact with the **KBase MCP API** at `https://hub.berdl.kbase.us/apis/mcp/`.

## How to use
1. Run all cells (Shift+Enter or **Run → Run All Cells**).
2. Enter your **KBase token** in the text box and press **Enter**.
3. The OpenAPI spec will be loaded and the available endpoints displayed.
4. Use the interactive caller at the bottom to choose an endpoint and send requests.

> **Note:** HTTP requests in this notebook use `pyodide-http` which patches Python's
> `requests` library to use the browser's `fetch` API. CORS must be enabled on the
> target server for requests to succeed.

In [ ]:
# ── Install runtime dependencies (Pyodide / JupyterLite) ───────────────────
import sys

IN_PYODIDE = sys.platform == 'emscripten'

if IN_PYODIDE:
    import micropip
    await micropip.install(['pyodide-http', 'ipywidgets'])
    import pyodide_http
    pyodide_http.patch_all()   # patch requests / urllib3 to use browser fetch
else:
    # Running in a standard Jupyter environment – install with pip if needed
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'requests', 'ipywidgets'])

print('Dependencies ready.')

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import json
import requests
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown, clear_output

# Base URL for the MCP API
BASE_URL = 'https://hub.berdl.kbase.us/apis/mcp'
OPENAPI_URL = f'{BASE_URL}/openapi.json'

print(f'API base URL : {BASE_URL}')
print(f'OpenAPI spec : {OPENAPI_URL}')

## Step 1 – Enter your KBase token

In [ ]:
# ── Token widget ─────────────────────────────────────────────────────────────
token_input = widgets.Password(
    placeholder='Paste your KBase token here …',
    description='KBase Token:',
    layout=widgets.Layout(width='550px'),
    style={'description_width': '120px'},
)

save_btn = widgets.Button(
    description='Save token',
    button_style='primary',
    icon='check',
)

token_status = widgets.Output()

KBASE_TOKEN = ''

def _on_save(b):
    global KBASE_TOKEN
    KBASE_TOKEN = token_input.value.strip()
    with token_status:
        clear_output()
        if KBASE_TOKEN:
            display(HTML('<span style="color:green">✓ Token saved (not shown for security).</span>'))
        else:
            display(HTML('<span style="color:red">✗ Token is empty.</span>'))

save_btn.on_click(_on_save)

display(widgets.HBox([token_input, save_btn]))
display(token_status)

## Step 1b – Create the API client

After saving your token above, run this cell to create a `hub_client` instance.
The client auto-discovers all available endpoints from the live OpenAPI spec.

Call `b.help()` at any time to list every method.

In [ ]:
# ── Create the hub_client ────────────────────────────────────────────────────
from hub_client import hub_client

b = hub_client(token=KBASE_TOKEN, url=BASE_URL)
b

In [ ]:
# ── Discover available methods ────────────────────────────────────────────────
b.help()

### Example calls

Edit and run the cells below to call any endpoint directly.

**Nested namespace access** (mirrors the API path):
```python
b.delta.databases.list()
b.delta.databases.tables.list(database="mydb")
b.delta.databases.tables.schema(database="mydb", table="mytable")
b.delta.databases.structure(with_schema=True)
b.delta.tables.count(database="mydb", table="mytable")
b.delta.tables.sample(database="mydb", table="mytable")
b.delta.tables.query.async_.submit(query="SELECT * FROM mydb.mytable LIMIT 10")
b.delta.tables.query.async_.status(job_id="<job_id>")
b.delta.tables.query.async_.results(job_id="<job_id>")
b.delta.tables.query.async_.jobs()
b.health()
```

**Flat method access** (all path segments joined with `_`):
```python
b.delta_databases_list()
b.delta_tables_query_async_submit(query="SELECT * FROM mydb.mytable LIMIT 10")
b.delta_tables_query_async_status(job_id="<job_id>")
```

In [ ]:
# ── List databases ────────────────────────────────────────────────────────────
result = b.delta.databases.list()
print(json.dumps(result, indent=2))

In [ ]:
# ── Submit an async query ─────────────────────────────────────────────────────
# Edit the SQL and run this cell, then check status/results below.

SQL = 'SELECT 1 AS hello'  # replace with your query

submission = b.delta.tables.query.async_.submit(query=SQL, limit=100)
job_id = submission['job_id']
print(f'Job submitted: {job_id}')

In [ ]:
# ── Poll job status ───────────────────────────────────────────────────────────
status = b.delta.tables.query.async_.status(job_id=job_id)
print(json.dumps(status, indent=2))

In [ ]:
# ── Fetch results (run after status is SUCCEEDED) ─────────────────────────────
results = b.delta.tables.query.async_.results(job_id=job_id)
print(json.dumps(results, indent=2))

## Step 2 – Load the OpenAPI specification

In [ ]:
# ── Fetch OpenAPI spec ───────────────────────────────────────────────────────

def get_headers():
    """Return request headers including the KBase token when available."""
    h = {'Accept': 'application/json'}
    if KBASE_TOKEN:
        h['Authorization'] = f'Bearer {KBASE_TOKEN}'
    return h


def load_openapi_spec(url: str = OPENAPI_URL) -> dict:
    """Load the OpenAPI spec, preferring the bundled local file.

    When running in JupyterLite the bundled openapi.json (written at build
    time) is loaded from the virtual filesystem, avoiding CORS issues.
    In a standard Jupyter environment the file is also used when present;
    otherwise the spec is fetched live from the remote URL.
    """
    import os
    local_path = 'openapi.json'
    if os.path.exists(local_path):
        with open(local_path) as f:
            return json.load(f)
    resp = requests.get(url, headers=get_headers(), timeout=30)
    resp.raise_for_status()
    return resp.json()


try:
    spec = load_openapi_spec()
    print(f"✓ OpenAPI spec loaded  –  title : {spec.get('info', {}).get('title', 'N/A')}")
    print(f"  version  : {spec.get('info', {}).get('version', 'N/A')}")
    print(f"  paths    : {len(spec.get('paths', {}))}")
except Exception as exc:
    spec = None
    print(f'✗ Failed to load spec: {exc}')
    print('Check your network connection and that the API is reachable.')

## Step 3 – Browse available endpoints

In [ ]:
# ── Display endpoint table ────────────────────────────────────────────────────

def display_endpoints(spec: dict):
    if spec is None:
        print('No spec loaded.')
        return

    paths = spec.get('paths', {})
    rows = []
    for path, methods in sorted(paths.items()):
        for method, op in methods.items():
            if method.startswith('x-'):
                continue
            summary = op.get('summary', op.get('operationId', ''))
            tags = ', '.join(op.get('tags', []))
            rows.append((method.upper(), path, summary, tags))

    html = [
        '<table style="border-collapse:collapse;width:100%;font-size:13px">',
        '<thead><tr>',
        *[f'<th style="border:1px solid #ccc;padding:6px 10px;background:#f0f0f0">{h}</th>'
          for h in ('Method', 'Path', 'Summary', 'Tags')],
        '</tr></thead><tbody>',
    ]
    METHOD_COLOR = {
        'GET': '#61affe', 'POST': '#49cc90', 'PUT': '#fca130',
        'DELETE': '#f93e3e', 'PATCH': '#50e3c2',
    }
    for method, path, summary, tags in rows:
        color = METHOD_COLOR.get(method, '#999')
        html.append(
            f'<tr>'
            f'<td style="border:1px solid #ddd;padding:5px 8px">'
            f'<span style="background:{color};color:#fff;border-radius:3px;padding:2px 6px;font-weight:bold">{method}</span></td>'
            f'<td style="border:1px solid #ddd;padding:5px 8px;font-family:monospace">{path}</td>'
            f'<td style="border:1px solid #ddd;padding:5px 8px">{summary}</td>'
            f'<td style="border:1px solid #ddd;padding:5px 8px">{tags}</td>'
            f'</tr>'
        )
    html.append('</tbody></table>')
    display(HTML(''.join(html)))


display_endpoints(spec)

## Step 4 – Interactive API caller

In [ ]:
# ── Build caller widgets ──────────────────────────────────────────────────────

def build_caller(spec: dict):
    if spec is None:
        print('No spec loaded – cannot build caller.')
        return

    paths = spec.get('paths', {})

    # Endpoint choices: "METHOD /path"
    choices = []
    for path, methods in sorted(paths.items()):
        for method in methods:
            if not method.startswith('x-'):
                choices.append(f'{method.upper()} {path}')

    # ── widgets ──────────────────────────────────────────────────────────────
    endpoint_dd = widgets.Dropdown(
        options=choices,
        description='Endpoint:',
        layout=widgets.Layout(width='600px'),
        style={'description_width': '100px'},
    )

    params_txt = widgets.Textarea(
        placeholder='Query / path parameters as JSON, e.g. {"limit": 10}',
        description='Params (JSON):',
        layout=widgets.Layout(width='600px', height='80px'),
        style={'description_width': '100px'},
    )

    body_txt = widgets.Textarea(
        placeholder='Request body as JSON (for POST/PUT/PATCH)',
        description='Body (JSON):',
        layout=widgets.Layout(width='600px', height='80px'),
        style={'description_width': '100px'},
    )

    call_btn = widgets.Button(
        description='Send request',
        button_style='success',
        icon='play',
        layout=widgets.Layout(width='160px'),
    )

    output = widgets.Output(
        layout=widgets.Layout(
            border='1px solid #ccc',
            padding='8px',
            min_height='100px',
            max_height='500px',
            overflow_y='auto',
        )
    )

    desc_out = widgets.Output()

    # ── helper: show endpoint description ────────────────────────────────────
    def _update_desc(change):
        val = change['new']
        if not val:
            return
        parts = val.split(' ', 1)
        method_key = parts[0].lower()
        path_key = parts[1] if len(parts) > 1 else ''
        op = paths.get(path_key, {}).get(method_key, {})
        with desc_out:
            clear_output()
            if op.get('description') or op.get('summary'):
                display(Markdown(
                    f"**{op.get('summary', '')}**\n\n{op.get('description', '')}"
                ))
            # show parameters
            params = op.get('parameters', [])
            if params:
                lines = ['| Name | In | Required | Type | Description |',
                         '|------|-----|----------|------|-------------|']
                for p in params:
                    schema = p.get('schema', {})
                    lines.append(
                        f"| `{p.get('name','')}` "
                        f"| {p.get('in','')} "
                        f"| {'✓' if p.get('required') else ''} "
                        f"| {schema.get('type','')} "
                        f"| {p.get('description','')} |"
                    )
                display(Markdown('\n'.join(lines)))

    endpoint_dd.observe(_update_desc, names='value')
    # trigger once
    _update_desc({'new': endpoint_dd.value})

    # ── helper: resolve $ref ─────────────────────────────────────────────────
    def _resolve_ref(ref_str: str, root: dict) -> dict:
        parts = ref_str.lstrip('#/').split('/')
        node = root
        for part in parts:
            node = node[part]
        return node

    # ── call handler ─────────────────────────────────────────────────────────
    def _on_call(b):
        val = endpoint_dd.value
        parts = val.split(' ', 1)
        method = parts[0].lower()
        path = parts[1] if len(parts) > 1 else ''

        # parse params
        params_raw = params_txt.value.strip()
        body_raw = body_txt.value.strip()
        try:
            params = json.loads(params_raw) if params_raw else {}
        except json.JSONDecodeError as e:
            with output:
                clear_output()
                print(f'✗ Invalid params JSON: {e}')
            return
        try:
            body = json.loads(body_raw) if body_raw else None
        except json.JSONDecodeError as e:
            with output:
                clear_output()
                print(f'✗ Invalid body JSON: {e}')
            return

        # build URL (substitute path parameters)
        url_path = path
        path_params = {k: v for k, v in params.items() if f'{{{k}}}' in path}
        for k, v in path_params.items():
            url_path = url_path.replace(f'{{{k}}}', str(v))
        query_params = {k: v for k, v in params.items() if k not in path_params}

        url = BASE_URL.rstrip('/') + url_path

        with output:
            clear_output()
            print(f'→ {method.upper()} {url}')
            if query_params:
                print(f'  query  : {json.dumps(query_params)}')
            if body is not None:
                print(f'  body   : {json.dumps(body)[:200]}…' if len(json.dumps(body)) > 200 else f'  body   : {json.dumps(body)}')
            try:
                fn = getattr(requests, method)
                kwargs = dict(headers=get_headers(), params=query_params, timeout=30)
                if body is not None:
                    kwargs['json'] = body
                resp = fn(url, **kwargs)
                print(f'← {resp.status_code} {resp.reason}')
                try:
                    data = resp.json()
                    print('\nResponse JSON:')
                    print(json.dumps(data, indent=2))
                except Exception:
                    print('\nResponse text:')
                    print(resp.text[:4000])
            except Exception as exc:
                print(f'✗ Request failed: {exc}')

    call_btn.on_click(_on_call)

    display(
        widgets.VBox([
            endpoint_dd,
            desc_out,
            params_txt,
            body_txt,
            call_btn,
            output,
        ])
    )


build_caller(spec)

## Step 5 – Raw API call helper

You can also call any endpoint directly in a code cell:

In [ ]:
# ── Example: call a specific endpoint ────────────────────────────────────────
# Edit the values below and run this cell.

ENDPOINT = '/'          # change to the path you want, e.g. '/tools'
METHOD   = 'get'        # 'get', 'post', 'put', 'delete', …
PARAMS   = {}           # query parameters dict
BODY     = None         # request body dict (for POST/PUT/PATCH)

url = BASE_URL.rstrip('/') + ENDPOINT
fn  = getattr(requests, METHOD)
kwargs = dict(headers=get_headers(), params=PARAMS, timeout=30)
if BODY is not None:
    kwargs['json'] = BODY

try:
    resp = fn(url, **kwargs)
    print(f'{resp.status_code} {resp.reason}')
    try:
        print(json.dumps(resp.json(), indent=2))
    except Exception:
        print(resp.text[:4000])
except Exception as exc:
    print(f'Error: {exc}')

## Step 6 – View full OpenAPI spec

In [ ]:
# Pretty-print the raw spec
if spec:
    print(json.dumps(spec, indent=2))
else:
    print('Spec not loaded.')